[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lbutler2405/EMP5027-rows-to-pixels/blob/main/notebooks/practical-6-deep-learning/EMP5027-Lecture-6a-Satellite-Land-Cover-Classification-EuroSAT.ipynb)


# EMP5027 Lecture 6a - Deep Learning I: Satellite Land-Cover Classification (EuroSAT)

*EMP5027, Methods in Data Analysis & Quality Assurance*

Dr. Liam Butler | Department of Systems & Control Engineering / Institute of Earth Systems, University of Malta


## Learning Objectives

By the end of this notebook you will be able to:
- Load and inspect an image classification dataset with `tensorflow_datasets`.
- Build a `tf.data` pipeline (resize, normalise, augment, batch, prefetch).
- Build and train a **Convolutional Neural Network (CNN) from scratch** in Keras.
- Apply **transfer learning** and **fine-tuning** with a pretrained ImageNet backbone.
- Evaluate an image classifier properly (classification report, confusion matrix, not just accuracy).
- Use **Grad-CAM** to visualise what a CNN is basing its predictions on.


## The story: mapping land cover from space

The SDM/geospatial lecture (Practical 5) trained classical ML models, logistic regression, random forest, SVM, MLP, on **raster values extracted at points**. This notebook tackles a related but distinct problem: classifying **entire image patches** cut from satellite imagery, where the spatial *pattern* across the whole patch (not just per-pixel values) carries the signal. That's exactly the kind of problem CNNs were built for.

We'll use **EuroSAT**: 27,000 Sentinel-2 satellite image patches (64×64 pixels, RGB) across 10 land-use/land-cover classes, forest, residential, industrial, river, pasture, and more. It's a standard benchmark in the remote-sensing deep-learning literature, small enough to train on a laptop, and a natural extension of the geospatial thread running through this course.


## Running this in Google Colab (recommended)

This notebook uses TensorFlow/Keras, which can be fiddly to install consistently across everyone's own laptops (GPU drivers, CUDA versions, conda vs. pip). **Google Colab** (colab.research.google.com) avoids all of that: TensorFlow is preinstalled, and a free GPU makes training several times faster.

**To use a GPU runtime:** `Runtime → Change runtime type → Hardware accelerator → GPU` (T4 is fine).

**Saving your work:** Colab sessions are temporary, and anything you don't explicitly save is lost when the runtime recycles. If you want to keep a trained model or exported results, mount your Google Drive (cell below) and save there, or download the file directly (`from google.colab import files; files.download("your_file")`).


In [ ]:
# --- Google Colab setup (safe to run locally too, it just skips these steps) ---
import sys

# sys.modules only contains "google.colab" when we're actually inside a Colab runtime,
# so this is a reliable way to branch behaviour without hardcoding a flag ourselves.
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # tensorflow_datasets isn't preinstalled on Colab; tensorflow itself already is.
    !pip install -q tensorflow_datasets

    import tensorflow as tf
    gpu_devices = tf.config.list_physical_devices("GPU")
    if gpu_devices:
        print(f"GPU runtime detected: {gpu_devices[0].name}, training will be fast.")
    else:
        print("No GPU detected. Go to Runtime > Change runtime type > GPU, then re-run this cell.")

    # Uncomment to save outputs (trained models, exported files) to your Google Drive:
    # from google.colab import drive
    # drive.mount('/content/drive')
else:
    print("Not running in Colab, assuming TensorFlow/tensorflow_datasets are already installed locally.")


## 0) Setup

We'll use **TensorFlow / Keras**, the same deep learning library used across all three notebooks in this set. If you're running this for the first time, install with:

```
pip install tensorflow tensorflow-datasets
```

A GPU (e.g. Google Colab's free GPU runtime) will make training much faster, but everything here is small enough to also run on a laptop CPU, it will just take a bit longer.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix

# Reproducibility: fixing the seeds means shuffling, weight initialisation and
# augmentation follow the same "random" sequence every run, so results are comparable
# from one run to the next rather than shifting purely from randomness.
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## 1) Load the Dataset

We load `eurosat` via `tensorflow_datasets` (TFDS). TFDS downloads the data once (cached locally) and hands it back as a `tf.data.Dataset` of `(image, label)` pairs, with no manual unzip or organise-into-folders step needed.


In [ ]:
DATASET_NAME = "eurosat"

# tfds.load fetches (and caches) the dataset, then returns three tf.data.Dataset
# objects, one per split, as (image, label) pairs (as_supervised=True). We carve our
# own train/val/test split out of the single "train" split TFDS ships, since EuroSAT
# doesn't come with an official train/val/test division.
(ds_train_raw, ds_val_raw, ds_test_raw), ds_info = tfds.load(
    DATASET_NAME,
    split=["train[:70%]", "train[70%:85%]", "train[85%:]"],
    as_supervised=True,
    with_info=True,
)

# ds_info carries the dataset's metadata, including the human-readable class names
# that correspond to each integer label.
CLASS_NAMES = ds_info.features["label"].names
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}):", CLASS_NAMES)
print("Train / Val / Test sizes:", ds_train_raw.cardinality().numpy(),
      ds_val_raw.cardinality().numpy(), ds_test_raw.cardinality().numpy())


In [ ]:
# Look at a grid of sample images with their labels
plt.figure(figsize=(10, 10))
for i, (image, label) in enumerate(ds_train_raw.take(9)):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image.numpy())
    plt.title(CLASS_NAMES[label.numpy()], fontsize=10)
    plt.axis("off")
plt.suptitle("Sample training images", y=1.02)
plt.tight_layout()
plt.show()


### Class balance

Before modelling anything, always check the label distribution.

In [ ]:
# .batch(256) here is just a convenient way to pull all the labels through the
# pipeline in chunks. We're not training on this batching, only reading labels.
labels = np.concatenate([y.numpy() for _, y in ds_train_raw.batch(256)])
counts = np.bincount(labels, minlength=NUM_CLASSES)

plt.figure(figsize=(9, 4))
sns.barplot(x=[CLASS_NAMES[i] for i in range(NUM_CLASSES)], y=counts)
plt.ylabel("Training images")
plt.xticks(rotation=45, ha="right")
plt.title("Class distribution (training split)")
plt.tight_layout()
plt.show()

for name, c in zip(CLASS_NAMES, counts):
    print(f"{name:>25s}: {c}")


## 2) Preparing the Data

Images arrive at different native resolutions/formats. We standardise them into a `tf.data` pipeline that:

1. **Resizes** every image to `96×96` pixels (a compromise between detail and training speed).
2. **Normalises** pixel values to `[0, 1]`.
3. **Batches** and **prefetches**, so the GPU/CPU is never left waiting on disk I/O.
4. Applies light **data augmentation** (flips/rotations) to the *training* set only, to reduce overfitting on a limited number of images, the same overfitting concern from the Occam's Razor practical, just in image form.


In [ ]:
IMG_SIZE = 96
BATCH_SIZE = 32

def preprocess(image, label):
    # Resize every image to a common size. EuroSAT images are already square, but
    # resizing here makes the pipeline robust to any source with mixed dimensions,
    # and fixes the input shape the CNN will expect.
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    # Cast to float32 and rescale pixel values from the raw 0-255 range down to [0, 1].
    # Neural networks train far more reliably on small, well-scaled inputs: large raw
    # pixel values push activations and gradients into ranges where training is slower
    # and less stable, the same reason we standardise features before fitting models
    # like logistic regression or an MLP.
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

# Data augmentation applies small, label-preserving random transformations (flips,
# a little rotation, a little zoom) to each training image on the fly. The model never
# sees the exact same pixels twice, which discourages it from memorising specific
# training images and encourages it to learn features that are robust to orientation
# and scale, both of which vary naturally in satellite imagery (there's no "correct"
# way up for a patch of forest or farmland). Note this is a Keras layer we call inside
# the model itself (see build_cnn below), not something we apply in the tf.data map
# below, and Keras automatically switches it off at evaluation/prediction time.
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="data_augmentation")

# AUTOTUNE lets tf.data pick sensible parallelism settings for us at runtime, rather
# than us guessing a fixed number of threads.
AUTOTUNE = tf.data.AUTOTUNE

# Training pipeline: map the preprocessing function across every image, shuffle so
# batches aren't dominated by whatever order the raw data happens to be in, batch into
# groups the model trains on at once, then prefetch so the next batch is being prepared
# on the CPU while the current batch is training on the GPU/CPU, instead of the
# accelerator sitting idle waiting for data.
train_ds = (ds_train_raw
            .map(preprocess, num_parallel_calls=AUTOTUNE)
            .shuffle(1000, seed=SEED)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

# Validation and test sets are preprocessed and batched the same way, but never
# shuffled: we want their order (and results) to be stable and reproducible, and
# shuffling would only add noise since we're not learning from these batches.
val_ds = (ds_val_raw
          .map(preprocess, num_parallel_calls=AUTOTUNE)
          .batch(BATCH_SIZE)
          .prefetch(AUTOTUNE))

test_ds = (ds_test_raw
           .map(preprocess, num_parallel_calls=AUTOTUNE)
           .batch(BATCH_SIZE)
           .prefetch(AUTOTUNE))

print("Batches, train:", train_ds.cardinality().numpy(),
      "| val:", val_ds.cardinality().numpy(),
      "| test:", test_ds.cardinality().numpy())


## 3) Part A - A Convolutional Neural Network From Scratch

Everything so far in this course has used **tabular** features: rows and columns. Images are different: nearby pixels are correlated, and the same pattern (an edge, a texture, a leaf vein) can appear anywhere in the frame. A plain `Dense` network flattens the image and throws that spatial structure away.

A **Convolutional Neural Network (CNN)** instead slides small learned filters across the image (`Conv2D`), keeping spatial relationships intact, and progressively downsamples (`MaxPooling2D`) to build up from edges, to textures, to parts, to whole-object patterns. This is the same "simple to complex, general to specific" idea from the model-comparison logic in the Occam's Razor practical, just applied through network depth rather than polynomial degree.

We'll build a small CNN, a few Conv/Pool blocks, then a classification head, and train it **from random initialisation** (no pretrained knowledge). This gives us an honest baseline before we bring in transfer learning in Part B.


In [ ]:
def build_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    # Augmentation lives inside the model graph: it only ever runs during training
    # (Keras handles switching it off automatically at evaluation/inference time), and
    # keeping it here means it travels with the model wherever it's used.
    x = data_augmentation(inputs)

    # Each block below is: Conv2D (learn a set of filters that each detect one local
    # pattern, e.g. an edge or a texture), BatchNormalization (rescales the activations
    # flowing through the network, which stabilises and speeds up training), then
    # MaxPooling2D (halves the spatial resolution by keeping only the strongest
    # response in each small window). Stacking these blocks lets the network build up
    # from simple local patterns in the early layers to increasingly complex,
    # larger-scale patterns in the later ones, while the number of filters grows
    # (32 → 64 → 128) to give it room to represent more of these complex patterns as
    # the spatial detail shrinks.
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    # GlobalAveragePooling2D collapses each of the 128 filter maps down to a single
    # number (its average activation), giving us a fixed-length feature vector
    # regardless of the input image size, ready to feed into a standard classification
    # head. Dropout randomly switches off a fraction of units during training, which
    # forces the network to not rely too heavily on any single unit and reduces
    # overfitting, the same idea in spirit as regularisation in a regression model.
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    # Softmax turns the final 10 outputs into a probability distribution over the 10
    # land-cover classes, so they sum to 1 and can be read directly as class
    # probabilities.
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="cnn_scratch")

cnn_model = build_cnn((96, 96, 3), NUM_CLASSES)
cnn_model.summary()


In [ ]:
cnn_model.compile(
    # Adam adapts its own learning rate per parameter as training goes, which makes it
    # a robust default optimiser that needs relatively little tuning to work well.
    optimizer=keras.optimizers.Adam(1e-3),
    # Sparse categorical crossentropy is the standard loss for multi-class
    # classification when labels are plain integers (0-9 here) rather than one-hot
    # encoded vectors, which is exactly the format TFDS gives us.
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    # Stop training once validation loss stops improving for 5 epochs in a row, and
    # roll back to the best-performing weights seen, rather than the weights from
    # whichever epoch happened to be last. This is our main defence against
    # overfitting: training accuracy can keep climbing long after the model has
    # stopped generalising.
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    # If validation loss plateaus for 2 epochs, halve the learning rate. A smaller
    # learning rate lets the model take finer steps once it's close to a good solution,
    # rather than overshooting it.
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]

EPOCHS = 20  # EarlyStopping will typically stop well before this

history_cnn = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)


In [ ]:
def plot_training_curves(history, title):
    # history.history stores the per-epoch loss/accuracy for both the training and
    # validation sets, which is exactly what we need to check for overfitting: if
    # training loss keeps falling while validation loss starts rising, the model is
    # starting to memorise the training set rather than learning generalisable patterns.
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(history.history["loss"], label="train")
    axes[0].plot(history.history["val_loss"], label="val")
    axes[0].set_title(f"{title}: Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history.history["accuracy"], label="train")
    axes[1].plot(history.history["val_accuracy"], label="val")
    axes[1].set_title(f"{title}: Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_training_curves(history_cnn, "CNN from scratch")


### Evaluating CNN from scratch

Accuracy alone can be misleading (especially with any class imbalance), so we also look at the **classification report** (precision/recall/F1 per class, the same metrics logic as the SDM notebook's sensitivity/specificity) and a **confusion matrix**.


In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred_probs = cnn_model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# labels=np.arange(NUM_CLASSES) keeps this well-defined even if the (smaller) test
# split happens not to contain every class.
print(classification_report(
    y_true, y_pred, labels=np.arange(NUM_CLASSES), target_names=CLASS_NAMES, zero_division=0
))


In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title(f"Confusion Matrix: CNN from scratch")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 4) Part B - Transfer Learning

Training a CNN from scratch on a few thousand images, as we just did, is working against a real handicap: the network has to relearn "what an edge is" and "what a texture is" from nothing. **Transfer learning** instead starts from a network already trained on millions of general images ([ImageNet](https://www.image-net.org/)) MobileNetV2 in our case, and reuses its learned low/mid-level visual features, only training a new classification head for our specific classes.

This is usually the single biggest lever for small-to-medium image datasets, and mirrors why the SDM notebook's ensemble beat any single model: borrowing strength from elsewhere pays off when your own labelled data is limited.

**Step 1, Feature extraction:** freeze the pretrained base, train only a new head.


In [ ]:
IMG_SIZE_TL = 96

# MobileNetV2 pretrained on ImageNet (1.4 million general-purpose images across 1000
# categories). include_top=False drops its original ImageNet classification head, so
# we get back just the convolutional "feature extractor" part, which already knows how
# to detect a huge range of general visual patterns (edges, textures, shapes) from
# having seen millions of images.
base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE_TL, IMG_SIZE_TL, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False  # freeze, we only train the new head for now

inputs = keras.Input(shape=(IMG_SIZE_TL, IMG_SIZE_TL, 3))
x = data_augmentation(inputs)
# Our tf.data pipeline already scaled pixels to [0, 1] (see preprocess above), but
# MobileNetV2 expects its own specific preprocessing (roughly rescaling to [-1, 1]).
# Rather than build a second data pipeline, we simply undo our /255 scaling here and
# hand raw 0-255 values to MobileNetV2's own preprocess_input, which applies whatever
# normalisation it was originally trained with.
x = layers.Rescaling(255.0)(x)  # undo our earlier /255 so we can use MobileNetV2's own preprocessing
x = keras.applications.mobilenet_v2.preprocess_input(x)
# training=False here keeps BatchNormalization layers inside the frozen base in
# inference mode, using their fixed running statistics from ImageNet training rather
# than recalculating statistics from our (much smaller) batches, which matters even
# though base_model.trainable is already False.
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

tl_model = keras.Model(inputs, outputs, name="mobilenetv2_transfer")
tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
tl_model.summary()


In [ ]:
# With the base frozen, only the new head (GlobalAveragePooling2D onward) has
# trainable weights, so this step is quick: we're training far fewer parameters than
# the scratch CNN, and starting from features that already work well for natural
# images rather than from random initialisation.
history_tl_head = tl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
)
plot_training_curves(history_tl_head, "Transfer learning: frozen base")


**Step 2, Fine-tuning:** unfreeze the top layers of the pretrained base and continue training with a much smaller learning rate, so we gently adapt those deeper, more task-specific features to our own images without wrecking what the network already knows.


In [ ]:
base_model.trainable = True

# Keep the earliest (most generic) layers frozen, and only fine-tune the later, more
# task-specific layers. Early layers in a CNN tend to learn very general features
# (edges, colours, simple textures) that are useful for almost any image task, while
# later layers learn increasingly dataset-specific combinations of those features.
# Unfreezing only the last 30 layers lets us adapt those higher-level features to
# EuroSAT's satellite imagery, without disturbing the generic low-level features that
# are already doing their job well.
FINE_TUNE_AT = len(base_model.layers) - 30
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),  # much smaller LR for fine-tuning
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# A learning rate 100x smaller than the head-training step matters here: the base
# model's weights already encode useful, hard-won knowledge from ImageNet, and large
# updates at this stage could wreck that knowledge (a problem sometimes called
# "catastrophic forgetting") faster than it can be usefully adapted to our data.
history_tl_finetune = tl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
)
plot_training_curves(history_tl_finetune, "Transfer learning: fine-tuned")


### Evaluating Transfer learning (fine-tuned)

Accuracy alone can be misleading (especially with any class imbalance), so we also look at the **classification report** (precision/recall/F1 per class, the same metrics logic as the SDM notebook's sensitivity/specificity) and a **confusion matrix**.


In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred_probs = tl_model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# labels=np.arange(NUM_CLASSES) keeps this well-defined even if the (smaller) test
# split happens not to contain every class.
print(classification_report(
    y_true, y_pred, labels=np.arange(NUM_CLASSES), target_names=CLASS_NAMES, zero_division=0
))


In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title(f"Confusion Matrix: Transfer learning (fine-tuned)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 5) Scratch CNN vs. Transfer Learning: Head to Head

With both models trained and evaluated, we put their held-out test accuracy side by side. This is the real comparison that matters: not which model trains faster or looks more sophisticated, but which one actually generalises better to satellite images it hasn't seen.


In [ ]:
def test_accuracy(model):
    _, acc = model.evaluate(test_ds, verbose=0)
    return acc

comparison = {
    "CNN from scratch": test_accuracy(cnn_model),
    "Transfer learning (fine-tuned)": test_accuracy(tl_model),
}

import pandas as pd
comp_df = pd.DataFrame(comparison.items(), columns=["Model", "Test accuracy"])
display(comp_df)

plt.figure(figsize=(5, 4))
sns.barplot(data=comp_df, x="Model", y="Test accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=15, ha="right")
plt.title("Held-out test accuracy")
plt.tight_layout()
plt.show()


## 6) Explainability: Where Is the Model Looking? (Grad-CAM)

A model that's merely *accurate* isn't necessarily one we should trust, the same concern that motivated variable importance in the SDM notebook and the OLS/GLM diagnostics in Lecture 4. **Grad-CAM** (Gradient-weighted Class Activation Mapping) highlights *which regions of an image* most influenced a CNN's prediction, by tracing the gradient of the predicted class back to the last convolutional layer.

This is a quick, qualitative sanity check: does the model focus on the parts of the image we'd expect a domain expert to look at, or has it latched onto something spurious (e.g. a watermark, the background, image borders)?


In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    # Build a helper model that outputs both the activations of the chosen
    # convolutional layer and the model's final predictions, so we can access both in
    # one forward pass.
    grad_model = keras.Model(model.inputs, [model.get_layer(last_conv_layer_name).output, model.output])
    # GradientTape records the operations run inside it, so we can later ask
    # TensorFlow for the gradient of the output with respect to something earlier in
    # the computation, here the chosen convolutional layer's activations.
    with tf.GradientTape() as tape:
        conv_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        # We only care about the score for one class (the predicted one), so we pull
        # out just that channel before computing gradients.
        class_channel = preds[:, pred_index]

    # This gradient tells us, for each location in the convolutional feature map, how
    # much increasing that location's activation would increase the predicted class's
    # score. Averaging it over height and width gives one "importance weight" per
    # filter channel: channels that matter more to this prediction get a bigger weight.
    grads = tape.gradient(class_channel, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    # Weighting each channel of the convolutional output by how important it was, then
    # summing across channels, produces a single spatial map: high values mark the
    # regions of the image that pushed the prediction towards the chosen class.
    heatmap = conv_output[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    # Clip negative values (regions that pushed away from the class, which we're not
    # interested in visualising here) and rescale to [0, 1] so the heatmap can be
    # plotted and compared consistently across images.
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)

# Find the last Conv2D layer in the scratch CNN automatically
last_conv_layer_name = next(l.name for l in reversed(cnn_model.layers) if isinstance(l, layers.Conv2D))
print("Using layer:", last_conv_layer_name)


In [ ]:
# Show Grad-CAM overlays for a handful of test images
import matplotlib.cm as cm

sample_images, sample_labels = next(iter(test_ds.unbatch().batch(8)))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    img = sample_images[i:i+1]
    heatmap, pred_idx = make_gradcam_heatmap(img, cnn_model, last_conv_layer_name)

    # The heatmap comes out at the (small) spatial resolution of the last
    # convolutional layer, so we resize it back up to the original image size before
    # overlaying it, and apply a "jet" colormap (blue = low importance, red = high) to
    # make it easy to read visually.
    heatmap_resized = tf.image.resize(heatmap[..., tf.newaxis], img.shape[1:3]).numpy().squeeze()
    heatmap_colored = cm.jet(heatmap_resized)[..., :3]

    # Blend the original image and the coloured heatmap together, so we can see both
    # the underlying satellite patch and where the model focused, at the same time.
    overlay = 0.6 * img[0].numpy() + 0.4 * heatmap_colored
    ax.imshow(np.clip(overlay, 0, 1))
    true_name = CLASS_NAMES[sample_labels[i].numpy()]
    pred_name = CLASS_NAMES[pred_idx]
    ok = "✓" if true_name == pred_name else "✗"
    ax.set_title(f"{ok} true: {true_name}\npred: {pred_name}", fontsize=9)
    ax.axis("off")

plt.suptitle("Grad-CAM: where the CNN is 'looking' for its prediction", y=1.02)
plt.tight_layout()
plt.show()


## 7) Looking at What the Model Gets Wrong

Grad-CAM told us where the model looks. It's just as useful to look directly at which images it gets wrong, and whether the mistakes make sense (visually similar land-cover types) or look arbitrary.


In [ ]:
mismatches = np.where(y_pred != y_true)[0]
print(f"{len(mismatches)} misclassified out of {len(y_true)} test images "
      f"({100*len(mismatches)/len(y_true):.1f}%)")

# Rebuild the full array of test images (unbatched, unshuffled, in the same order as
# y_true/y_pred) so we can index directly into it by mismatch position.
sample_images_all = np.concatenate([x.numpy() for x, _ in test_ds])

if len(mismatches) > 0:
    show_idx = mismatches[:8]
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax, idx in zip(axes.flat, show_idx):
        ax.imshow(sample_images_all[idx])
        ax.set_title(f"true: {CLASS_NAMES[y_true[idx]]}\npred: {CLASS_NAMES[y_pred[idx]]}", fontsize=9)
        ax.axis("off")
    for ax in axes.flat[len(show_idx):]:
        ax.axis("off")
    plt.suptitle("Sample misclassifications", y=1.02)
    plt.tight_layout()
    plt.show()


## 8) Practice Exercises

1. **Deeper CNN:** add a fourth `Conv2D`/`MaxPooling2D` block to `build_cnn`. Does it improve validation accuracy, or start to overfit?
2. **Different backbone:** swap `MobileNetV2` for `keras.applications.EfficientNetB0` (remember to use its matching `preprocess_input`). Compare accuracy and training time.
3. **Fewer fine-tuned layers:** change `FINE_TUNE_AT` to unfreeze only the last 10 layers instead of 30. How does this affect the fine-tuning curves?
4. **Confusable classes:** looking at the confusion matrix, which two land-cover classes does the model confuse most? Why might that make physical sense (e.g. visually similar from above)?
5. **Grad-CAM audit:** find a *correct* prediction where the Grad-CAM heatmap looks like it's focusing on a sensible region, and an *incorrect* one where it looks off. Write a sentence on each.


## 9) Wrap-Up and Next Steps

You've now trained and compared two deep learning approaches to satellite image classification, and used Grad-CAM to check *why* the model predicts what it predicts, not just *whether* it's right.

**Next steps / extensions:**
- Move beyond RGB to EuroSAT's **multispectral (13-band Sentinel-2)** variant. Real remote-sensing pipelines rarely stop at visible light.
- Try **semantic segmentation** (pixel-level land-cover maps) rather than whole-patch classification, a natural next step once patch-level classification works well.
- Combine this CNN's patch-level view with the SDM notebook's point-level raster extraction: could patch classification features improve the SDM's environmental predictors?
- Explore **spatial train/test splits** (as flagged in the SDM notebook) to check the model isn't just memorising specific regions.
